In [1]:
from langchain_core.tools import tool
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os


load_dotenv(override=True)

True

In [2]:
token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not token:
    print("  ERROR: Token not found! Check your .env file.")
else:
    print("  Token loaded successfully.")

  Token loaded successfully.


In [11]:
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-4-31B-it",
    task="text-generation",
)
#deepseek-ai/DeepSeek-R1-0528
chat = ChatHuggingFace(llm=llm)

In [12]:
response = chat.invoke("What is LangGraph?")


print(response)

content='**LangGraph** is a library developed by the LangChain team designed to build **stateful, multi-agent applications** using Large Language Models (LLMs).\n\nWhile standard LangChain is great for creating "chains" (a linear sequence of steps), LangGraph is designed for **cyclic workflows**. In simpler terms, it allows you to create AI agents that can loop back, correct their own mistakes, and maintain a complex memory of the conversation.\n\nHere is a detailed breakdown of what makes LangGraph different and why it is useful.\n\n---\n\n### 1. The Core Problem it Solves: Linear vs. Cyclic\nMost LLM frameworks use a **Directed Acyclic Graph (DAG)**. This means the data flows in one direction:\n`Input` $\\rightarrow$ `Prompt` $\\rightarrow$ `LLM` $\\rightarrow$ `Output`.\n\nHowever, real-world reasoning is rarely linear. A sophisticated agent needs to:\n*   **Reason:** "I need to find the weather in Tokyo."\n*   **Act:** Call a weather API.\n*   **Observe:** "The API returned an erro

In [13]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`. """


    return a * b

@tool
def subtract(a: int, b: int) -> int:
    """Subtract `b`from `a`. """

    return a -b

@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.     """
    return a / b

tools= [add,multiply,subtract,divide]
tools_by_name = {tool.name: tool for tool in tools}

#mode_with_tools =ChatHuggingFace(llm=llm,tools=tools_by_name)
llm_with_tools = chat.bind_tools(tools)


In [16]:
# response=llm_with_tools.invoke("What is 2+3? and use the tools given to you only to answer the questions")

from langchain_core.messages import HumanMessage

response = llm_with_tools.invoke(
    [HumanMessage(content="""You MUST use a tool to solve the problem.DO NOT compute yourself.If you do not use a tool, the answer is wrong.
    Question:What is 2+3?""" )])

In [17]:
print(response)

content='' additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 2, "b": 3}', 'name': 'add', 'description': None}, 'id': 'chatcmpl-tool-a19fda25eec5b887', 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 273, 'total_tokens': 288}, 'model_name': 'google/gemma-4-31B-it', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d59c9-397e-7773-b5e3-473dc43df4b1-0' tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'chatcmpl-tool-a19fda25eec5b887', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 273, 'output_tokens': 15, 'total_tokens': 288}


In [21]:
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
args = tool_call["args"]

result = tools_by_name[tool_name].invoke(args)

print("Tool result:", result)

Tool result: 5
